In [ ]:
# Copyright 2026 Arm Limited and/or its affiliates.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

# WIP: TOSA/EthosU/VgfQuantizer composable quantizer tutorial

This is an in-depth tutorial of the new `TOSA/EthosU/VgfQuantizer` API. While the `TOSAQuantizer` is used in the example, both the
`EthosUQuantizer` and `VgfQuantizer` directly inherit from this base class. 

Note that the main API and functionality remains largely the same to allow for a drop-in replacement, but the underlying framework is different - as will be explained. **Both the quantizer and this tutorial are currently experimental and may change without prior notice.** Refer to https://github.com/pytorch/executorch/issues/17701 for questions and feedback.

Before you begin:
1. (In a clean virtual environment with a compatible Python version) Install executorch using `./install_executorch.sh`
2. Install Arm TOSA dependencies using `examples/arm/setup.sh --disable-ethos-u-deps`

With all commands executed from the base `executorch` folder.

## Setup model and logging

In [ ]:
import torch

class ToyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 1, 1)
        self.conv2 = torch.nn.Conv2d(1, 1, 1)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = torch.relu(x)        
        y = self.conv2(y)
        z = x/y
        return z.view((1,))

example_inputs = (torch.ones(1,1,1,1),torch.ones(1,1,1,1))

model = ToyModel()

In [ ]:
# Set logger to DEBUG for full quantization report
import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.DEBUG)

In [ ]:
# If you have model-explorer installed, you can visualize the exported program with the following code:
from executorch.devtools.visualization import visualize

exported_program = torch.export.export(model, example_inputs)
visualize(exported_program)

# Basic quantizer useage
The experimental API is enabled by setting `use_composable_quantizer=True` when initializing
the quantizer. The name `composable_quantizer` refers to the new implementation using multiple
separate quantizers; the user configures quantization by specifying a sequence of quantizers,
with each annotating a selection of nodes with a particular quantization config.

The node selection and quantization config is set by the user. They can be selected using basic API-calls as demonstrated here,
or be completely customized as shown in the advanced section. However, the backend has limits on what is supported and may
reject quantization of nodes with unsupported quantization configs. A few operators additionally require special quantization
strategies for numerical correctness, which is encoded in the backend specific `TOSAQuantizationSpec`. These special cases will
be reported in a quantization report.

The quantizer additionally applies it's own filtering of the selected nodes to only quantize what is known to be supported
in the backend.

Below, the model is quantized by three different quantizers:
1. The nodes named 'conv2d' and 'relu' are quantized using the a8a8 config.
2. The remaning conv is targeted with None, leaving it non-quantized.
3. The remaining nodes are targeted by the global config, which is a16w8.

Note that order of configuration is important, later specified quantizers have precedence (with the exception of global,
which is always applied last). Switching 1 and 2 would leave both convolutions in floating point. 

In [ ]:
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.quantizer import (
    TOSAQuantizer,
    get_symmetric_quantization_config,
    get_symmetric_a16w8_quantization_config,
)
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e

# Export the model
exported_program = torch.export.export(model, example_inputs)
graph_module = exported_program.module(check_guards=False)

# Create and configure quantizer to use a symmetric quantization config globally on all nodes
target = "TOSA-1.0+INT"
compile_spec = TosaCompileSpec(target)
quantizer = TOSAQuantizer(compile_spec, use_composable_quantizer=True)

a16w8_config = get_symmetric_a16w8_quantization_config()
fp_config = None
a8w8_config = get_symmetric_quantization_config()

quantizer.set_global(a16w8_config)                        # Gloabl config, applied last
quantizer.set_module_type(torch.nn.Conv2d, fp_config)     # Applied second, remaning conv2d set to floating point
quantizer.set_node_name(["conv2d", "relu"], a8w8_config)  # Applied first, conv+relu quantized using the a8a8 config

# Post training quantization
quantized_graph_module = prepare_pt2e(graph_module, quantizer)
quantized_graph_module(*example_inputs) # Calibrate the graph module with the example input
quantized_graph_module = convert_pt2e(quantized_graph_module)

In [ ]:
exported_program = torch.export.export(quantized_graph_module, example_inputs)
visualize(exported_program)

### The quantization report

In the logged quantization report each quantizer has added one header describing targeted nodes, the used quantization config, and the supported operators / operator patterns. 
```
PatternQuantizer using NodeNameNodeFinder targeting names: conv2d, relu
Annotating with executorch.backends.arm.quantizer.arm_quantizer.get_symmetric_quantization_config(is_per_channel=True)
Supported operators and patterns defined by executorch.backends.arm.quantizer.quantizer_support.TOSA_QUANTIZER_SUPPORT_DICT
```

it then gives a short overview of how many nodes it has targeted
```
   Accepted nodes: 2
   Rejected due to previous annotation: 0
   Rejected nodes: 0
```

and finally a node-by-node report
```
       NODE NAME    INPUT QSPEC MAP                           OUTPUT QSPEC MAP
   --  -----------  ----------------------------------------  ---------------------
   ╒   conv2d       x: INT8_PER_TENSOR_QSPEC                  NO_QSPEC
   |                _param_constant0: INT8_PER_CHANNEL_QSPEC
   |                _param_constant1: DERIVED_QSPEC
   ╘   relu  
```

The brackets here indicates that the conv2d and relu has been recognized as a single pattern
to be quantized to allow a fusing later in the backend. One quantization config translates to
many different quantization annotations for different types of tensors; per tensor for
activations, per channel for weights, and a special quantization spec for the int32 bias. 

### Pre-transform for annotation vs. final quantization report
One important detail is that there are two reports printed, one named PRE-TRANSFORM_FOR_ANNOTATION QUANTIZATION REPORT,
and one named FINAL QUANTIZATION REPORT. This is related to the fact that some operators has to be decomposed before quantization to ensure
that all "sub operators" gets quantized properly. As an example, the division operator in the first report
has decomposed into a reciprocal and multiplication operator in the second. Had it not been marked for quantization
in the first step, it would have remained a single division operator.

**This is important to be aware of when doing mixed quantization since this means that for an operator to be fully quantized,
both the original operator and the decomposition needs to be targeted.**

### SharedQspecQuantizer
Last in the report there is always an additional quantizer applied which is not specified by the user, the SharedQspecQuantizer.
It handles data shuffling operators without numerical behaviour such as copies and reshapes to ensure that they are quantized with the same qspec as
surrounding nodes, rather than counting on the user to configure them correctly. It shouldn't need much attention as it is not configured, 
but it is good to be aware of when analyzing the quantization behaviour. The targeted operators are defined by `SHARED_QSPEC_OPS_DEFAULT`
in the quantizer class.

# Advanced quantizer useage

The composability of the quantizer has an additional benefit for advanced user in that each component can easily be modified
or completely be swapped out in cases where special behaviour is needed. Let's see this in action by recreating what happens under
the hood when the `set_node_name` API is used.

In [ ]:
import executorch.backends.cortex_m.quantizer.node_finders as node_finders
from executorch.backends.arm.quantizer.quantization_config import TOSAQuantizationConfig
from executorch.backends.arm.quantizer.arm_quantizer_utils import PatternQuantizer
from executorch.backends.cortex_m.quantizer.pattern_matcher import PatternMatcher
from torchao.quantization.pt2e.quantizer import (
    QuantizationSpec,
)
from torchao.quantization.pt2e import (
    MinMaxObserver,
)

# Export the model
exported_program = torch.export.export(model, example_inputs)
graph_module = exported_program.module(check_guards=False)

# Create and configure quantizer to use a symmetric quantization config globally on all nodes
target = "TOSA-1.0+INT"
compile_spec = TosaCompileSpec(target)
quantizer = TOSAQuantizer(compile_spec, use_composable_quantizer=True)


# The first component is the selection of nodes, done through NodeFinders
# A node finder is a class implementing the NodeFinder interface
# This is instantiated inside the set_node_name function
node_finder = node_finders.NodeNameNodeFinder("conv2d")

# The second component is the quantization config, which may be custom
# This is what is returned by the get_symmetric_quantization_config function
qspec = QuantizationSpec(torch.int8, MinMaxObserver, quant_min=-128, quant_max=127, qscheme = torch.per_tensor_symmetric)
quantization_config = TOSAQuantizationConfig(input_activation=qspec,output_activation=qspec, weight=None, bias=None)

# The third component is the pattern matcher which defines support for the backend
# This would typically be the TOSA_QUANTIZER_SUPPORT_DICT but here a minimal support dict is created for demonstration purposes
# A pattern_checker is a class implementing the PatternChecker interface, or it can be None if no extra checks are needed
# This is instantiated by the backend and used by all sub-quantizers
pattern_checker = None
SUPPORT_DICT = {(torch.ops.aten.conv2d.default,) : pattern_checker}
pattern_matcher = PatternMatcher(SUPPORT_DICT, "MY_SUPPORT_DICT")

# All components are brought together in the PatterQuantizer and added to the quantizer
# This is done last in the set_node_name function
pattern_quantizer = PatternQuantizer(quantization_config, node_finder, pattern_matcher)
quantizer.add_quantizer(pattern_quantizer)

quantized_graph_module = prepare_pt2e(graph_module, quantizer)
quantized_graph_module(*example_inputs)
quantized_graph_module = convert_pt2e(quantized_graph_module)

As confirmed by the report, the quantizer has now only targeted one single convolution with a custom quantization config, and this config was then propagated to the relu node by the SharedQpsecQuantizer as expected. The view stays in float since the preceeding division operator is in float.

This useage of the quantizer has less guarantees of producing numerically correct or even functional graphs, but it can be a useful tool for debugging or when an otherwise unsupported behaviour is required.